## Imports

In [0]:
import requests
import zipfile
import os

## Setting Path for Downloading Dataset

In [0]:
catalog, schema, volume = "bts_flight_data", "bronze", "bts_flight_dataset"
volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
zip_dir = f"{volume_path}/zipped"
extract_dir = f"{volume_path}/unzipped"

os.makedirs(zip_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)

year = 2025

## Running Loop to Download All Files for 2025

In [0]:
for month in range(1, 13):
    url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
    dest = f"{zip_dir}/flights_{year}_{month:02d}.zip"

    if os.path.exists(dest):
        print(f"Already downloaded: {dest}")
        continue

    resp = requests.get(url, verify=False, stream=True)
    if resp.status_code == 200:
        with open(dest, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
        print(f"Downloaded month {month}")
    else:
        print(f"Month {month} failed: {resp.status_code}")

## Unzipping All Files

In [0]:
for filename in os.listdir(zip_dir):
    if filename.endswith(".zip"):
        zip_path = os.path.join(zip_dir, filename)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        print(f"Extracted {filename}")

## Reporting Size Before and After Unzipping

In [0]:
def get_folder_size(path):
    total_bytes = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_bytes += os.path.getsize(fp)
    return total_bytes

zip_size = get_folder_size(zip_dir)
extracted_size = get_folder_size(extract_dir)

print(f"\nZipped size:     {zip_size / (1024**3):.2f} GB")
print(f"Extracted size:  {extracted_size / (1024**3):.2f} GB")
if zip_size > 0:
    print(f"Expansion ratio: {extracted_size / zip_size:.1f}x")

## Loading Unzipped CSV files into Bronze Delta Table

In [0]:
df = spark.read.csv(f"{extract_dir}/*.csv", header=True, inferSchema=True)
df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.flights_raw")
print(f"Row count: {df.count()}")

In [0]:
df.display()